# AEV-PLIG Results Analysis
Multi-model accuracy · ensemble agreement · uncertainty calibration · per-target ranking · outliers

In [ ]:
import re

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

from aev_plig import results

In [ ]:
PREDICTIONS_DIR      = "output/predictions"
# Stem of the parquet file produced by scripts/predict.py, e.g.:
# "pdbbind_U_bindingnet_U_bindingdb_ligsim90_fep_benchmark_predictions"
PREDICTION_FILE_NAME = "fep_benchmark_predictions"
TRUTH_COL            = "pK"
PRED_COL             = "preds"   # ensemble-average column
UID_COL              = "unique_id"
TOP_N_OUTLIERS       = 20
MIN_TARGET_SAMPLES   = 2
FIG_DIR              = None      # set to a Path to auto-save HTML figures

In [ ]:
df = results.load_all_predictions(
    PREDICTIONS_DIR, prediction_file_name=PREDICTION_FILE_NAME
)

# Auto-detect ensemble member columns (preds_0, preds_1, ...)
pred_member_cols = sorted(
    [c for c in df.columns if re.fullmatch(r"preds_\d+", c)],
    key=lambda c: int(c.split("_")[1]),
)
n_models = len(pred_member_cols)
model_names = (
    df["model_name"].unique().to_list() if "model_name" in df.columns else "N/A"
)
print(
    f"Loaded {df.height:,} rows | {n_models} ensemble members | "
    f"model_name values: {model_names}"
)

# Ensemble uncertainty = std dev across member predictions
if n_models > 1:
    df = df.with_columns(
        pl.concat_list([pl.col(c) for c in pred_member_cols])
        .list.std()
        .alias("uncertainty")
    )
else:
    df = df.with_columns(pl.lit(0.0).alias("uncertainty"))

# Residual (requires ground-truth column)
if TRUTH_COL in df.columns:
    df = df.with_columns(
        (pl.col(PRED_COL) - pl.col(TRUTH_COL)).alias("residual")
    )

df.head(3)

## §1 Data Overview

In [ ]:
clean = df.filter(pl.col(TRUTH_COL).is_not_null())
fig = px.histogram(
    clean.to_pandas(),
    x=TRUTH_COL,
    nbins=40,
    color="model_name" if "model_name" in clean.columns else None,
    barmode="overlay",
    opacity=0.7,
    title=f"Distribution of true {TRUTH_COL}  (n={clean.height:,})",
)
fig.show()
print(clean.select(pl.col(TRUTH_COL)).describe())

## §2 Ensemble Member Metrics
One row per ensemble member + the ensemble average, grouped by `model_instance`.

In [ ]:
extra_cols = ["model_instance"] if "model_instance" in df.columns else []
clean_pd = (
    df.select([TRUTH_COL, PRED_COL] + pred_member_cols + extra_cols)
    .drop_nulls()
    .to_pandas()
)

rows = []
groups = (
    clean_pd.groupby("model_instance")
    if "model_instance" in clean_pd.columns
    else [(None, clean_pd)]
)
for grp_val, grp in groups:
    y_true = grp[TRUTH_COL].values
    for col in pred_member_cols + [PRED_COL]:
        y_pred = grp[col].values
        label = "Ensemble avg" if col == PRED_COL else col
        rows.append(
            {
                "model_instance": grp_val or "",
                "Model": label,
                "RMSE": round(results.rmse(y_true, y_pred), 4),
                "Pearson R": round(results.pearson_r(y_true, y_pred), 4),
                "Kendall τ": round(results.kendall_tau(y_true, y_pred), 4),
            }
        )

metrics_df = pd.DataFrame(rows)
display(
    metrics_df.style.highlight_min(subset=["RMSE"], color="lightgreen").highlight_max(
        subset=["Pearson R", "Kendall τ"], color="lightgreen"
    )
)

fig = px.bar(
    metrics_df,
    x="Model",
    y="RMSE",
    color="model_instance" if "model_instance" in metrics_df.columns else None,
    barmode="group",
    title="RMSE per ensemble member vs ensemble average",
)
fig.show()

## §3 Predicted vs True
Colour and error bars show ensemble standard deviation.

In [ ]:
extra_cols = ["model_instance"] if "model_instance" in df.columns else []
plot_df = (
    df.select([TRUTH_COL, PRED_COL, UID_COL, "uncertainty"] + extra_cols)
    .drop_nulls()
    .to_pandas()
)

x_range = [plot_df[TRUTH_COL].min(), plot_df[TRUTH_COL].max()]
fig = px.scatter(
    plot_df,
    x=TRUTH_COL,
    y=PRED_COL,
    color="uncertainty",
    color_continuous_scale="Viridis",
    error_y="uncertainty",
    facet_col="model_instance" if "model_instance" in plot_df.columns else None,
    hover_data=[UID_COL],
    title="Predicted vs True  (colour/error bars = ensemble std dev)",
)
instances = (
    plot_df["model_instance"].unique().tolist()
    if "model_instance" in plot_df.columns
    else [None]
)
for _ in instances:
    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=x_range,
            mode="lines",
            line=dict(dash="dash", color="black"),
            showlegend=False,
        )
    )
fig.show()

## §4 Residual Analysis

In [ ]:
res_df = df.select([TRUTH_COL, "residual", UID_COL]).drop_nulls().to_pandas()
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Residual distribution", f"Residual vs True {TRUTH_COL}"],
)
fig.add_trace(
    go.Histogram(x=res_df["residual"], nbinsx=40, name="Residuals"), row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=res_df[TRUTH_COL],
        y=res_df["residual"],
        mode="markers",
        text=res_df[UID_COL],
        marker=dict(opacity=0.5),
        name="Residual",
    ),
    row=1,
    col=2,
)
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(title="Residual Analysis", showlegend=False)
fig.show()

## §5 Uncertainty Calibration
Does ensemble std dev predict prediction error?

In [ ]:
if n_models > 1:
    cal_df = df.select(["uncertainty", "residual"]).drop_nulls().to_pandas()
    cal_df["abs_residual"] = cal_df["residual"].abs()
    r = np.corrcoef(cal_df["uncertainty"], cal_df["abs_residual"])[0, 1]
    fig = px.scatter(
        cal_df,
        x="uncertainty",
        y="abs_residual",
        opacity=0.4,
        title=f"Uncertainty calibration  (Pearson R = {r:.3f})",
        labels={"uncertainty": "Ensemble std dev", "abs_residual": "|Residual|"},
    )
    xs = np.sort(cal_df["uncertainty"].values)
    fig.add_trace(
        go.Scatter(
            x=xs,
            y=np.poly1d(np.polyfit(cal_df["uncertainty"], cal_df["abs_residual"], 1))(
                xs
            ),
            mode="lines",
            name="Linear fit",
            line=dict(color="red"),
        )
    )
    fig.show()
else:
    print("Single model — uncertainty calibration requires multiple ensemble members.")

## §6 Per-Target Kendall τ (Ranking Ability)
Each `unique_id` is treated as its own target.

In [ ]:
target_metrics = results.per_target_metrics(
    df,
    target_col=UID_COL,
    truth_col=TRUTH_COL,
    pred_col=PRED_COL,
    min_samples=MIN_TARGET_SAMPLES,
)
fig = px.histogram(
    target_metrics.to_pandas(),
    x="kendall_tau",
    nbins=30,
    title="Per-target Kendall τ distribution",
    labels={"kendall_tau": "Kendall τ"},
)
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.show()
display(target_metrics.sort("kendall_tau").to_pandas())

## §7 Outlier Table

In [ ]:
outlier_cols = [UID_COL, TRUTH_COL, PRED_COL, "residual", "uncertainty"]
if "model_instance" in df.columns:
    outlier_cols.insert(1, "model_instance")
outliers = (
    df.select(outlier_cols)
    .drop_nulls()
    .with_columns(pl.col("residual").abs().alias("abs_residual"))
    .sort("abs_residual", descending=True)
    .head(TOP_N_OUTLIERS)
)
display(outliers.to_pandas())

## §8 Success Rate

In [ ]:
res_vals = df.select("residual").drop_nulls()["residual"].to_numpy()
success_rows = [
    {
        "Threshold (\u00b1pK)": t,
        "N within": int((np.abs(res_vals) <= t).sum()),
        "% within": round(100 * (np.abs(res_vals) <= t).mean(), 1),
    }
    for t in [0.5, 1.0, 1.5, 2.0]
]
display(pd.DataFrame(success_rows))